
# Árboles y ensambles, Notebook 5
## Importancia de variables: qué significa que un ratio "importe", en cada modelo y comparado con los betas

**Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De qué se trata

Después de entrenar cualquier modelo, la pregunta que viene del negocio es siempre la misma:
**¿qué variables pesan?** Un gerente quiere saber si lo que decide el riesgo es la liquidez o el
endeudamiento; un regulador quiere saber que el modelo no se apoya en algo indebido. Cada familia
de modelos responde de una forma distinta, y en este notebook ponemos las respuestas lado a lado,
calculadas con los mismos datos y la misma partición de los notebooks anteriores:

| Modelo | Qué entrega | Tiene signo | Supone linealidad |
|---|---|---|---|
| Regresión lineal o logística | coeficientes $\beta_j$ | sí | sí |
| Árbol (ID3, CART) | importancia por **impureza**: cuánta impureza redujo cada variable | no | no |
| Random forest, boosting | la misma, promediada o acumulada sobre los árboles | no | no |
| Cualquier modelo | importancia por **permutación**: cuánto empeora si se baraja la variable | no | no |
| Cualquier modelo | contribuciones por observación (**SHAP**) | sí, por observación | no |

### Qué vas a aprender hoy

1. Qué mide un coeficiente de regresión y por qué hay que estandarizar para comparar.
2. Cómo se calcula la importancia por impureza de un árbol, nodo por nodo, a mano y con `scikit-learn`.
3. Cómo se promedia en un bosque, y un sesgo conocido de esa medida.
4. La importancia por permutación, que sirve para cualquier modelo.
5. Qué reportan AdaBoost y gradient boosting.
6. SHAP: recuperar el signo, empresa por empresa.
7. Cómo leer las diferencias entre las medidas sin concluir que alguna está mal.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

import statsmodels.api as sm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Listo.")



## 1. Los datos, otra vez

Las mismas 219 empresas y la misma partición (146 de entrenamiento, 73 de prueba).


| Ratio | Qué mide |
|---|---|
| `deuda_activos` | deuda total / activos totales (endeudamiento) |
| `razon_corriente` | activo circulante / pasivo circulante (liquidez) |
| `ventas_deuda` | ventas / deuda total (capacidad de servir la deuda) |
| `ln_activos` | logaritmo de los activos totales (tamaño) |
| `roa` | utilidad neta / activos totales (rentabilidad) |
| `impago` | **lo que queremos predecir:** 1 si la empresa cayó en impago, 0 si no |


In [ ]:

# Las 219 empresas con ratios financieros (la tabla de impago del curso, con 5 de sus ratios).
# Está pegada aquí mismo para que el notebook no dependa de ningún archivo externo.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Empresas:", len(impago), "  Fracción en impago:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# La misma partición en todos los notebooks del set: 146 empresas para entrenar, 73 para probar
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Entrenamiento:", len(X_train), "  Prueba:", len(X_test))
print("Fracción en impago: entrenamiento", round(y_train.mean(), 3), " prueba", round(y_test.mean(), 3))



## 2. La respuesta clásica: los betas de una regresión

Para una respuesta de sí o no, la regresión logística modela el logaritmo de la razón de
probabilidades (el *log-odds*):

$$
\ln \frac{p}{1 - p} = \beta_0 + \beta_1 x_1 + \dots + \beta_5 x_5
$$

donde $p$ es la probabilidad de impago, $x_j$ son los ratios y $\beta_j$ sus coeficientes. Un
$\beta_j$ positivo dice que el ratio sube el riesgo; negativo, que lo baja. Cada coeficiente se
lee **manteniendo los demás ratios fijos**.

Un coeficiente tiene unidades: el de la razón corriente está "por unidad de razón corriente" y
el de `ln_activos` "por unidad de logaritmo", así que sus tamaños no se pueden comparar. Para
compararlos se **estandarizan** las variables (se les resta el promedio y se dividen por la
desviación estándar, calculados solo con el entrenamiento) y entonces cada $\beta$ se lee como el
efecto de moverse una desviación estándar.

Los coeficientes se estiman por máxima verosimilitud, que es exactamente minimizar la entropía
cruzada del Notebook 1. `statsmodels` entrega además el error estándar de cada coeficiente, y
con él un valor $z$ y un valor $p$ que dicen si el coeficiente se distingue de cero.


In [ ]:

mu, sd = X_train.mean(), X_train.std(ddof=0)
Z_train = (X_train - mu) / sd
Z_test = (X_test - mu) / sd

logit = sm.Logit(y_train, sm.add_constant(Z_train)).fit(disp=0)
tabla_betas = pd.DataFrame({"beta estandarizado": logit.params, "error estándar": logit.bse, "z": logit.tvalues, "valor p": logit.pvalues}).round(3)
tabla_betas["lectura"] = np.where(logit.params > 0, "más alto, más riesgo", "más alto, menos riesgo")
print(tabla_betas)
print()
p_test = logit.predict(sm.add_constant(Z_test))
print(f"Acierto en prueba (umbral 0,5): {accuracy_score(y_test, (p_test > 0.5).astype(int)):.3f}   AUC: {roc_auc_score(y_test, p_test):.3f}")


In [ ]:

betas = tabla_betas.drop("const").sort_values("beta estandarizado", key=abs)
fig = px.bar(betas, x="beta estandarizado", y=betas.index, orientation="h", error_x="error estándar",
             color=(betas["beta estandarizado"] > 0).map({True: "sube el riesgo", False: "baja el riesgo"}),
             color_discrete_map={"sube el riesgo": "#B4562A", "baja el riesgo": "#3B8A55"},
             title="Betas estandarizados de la regresión logística (con su error estándar)")
fig.update_layout(yaxis_title="", legend_title="", height=380)
fig.show()



**Cómo leer la tabla.** Según la regresión, el ratio que más mueve el riesgo por desviación
estándar es la razón corriente, con signo negativo: más liquidez, menos impago. Fíjate en dos
cosas. Primero, con 146 empresas los errores estándar son grandes y varios coeficientes no se
distinguen de cero (valor $p$ mayor que 0,10). Segundo, el signo de `roa` es positivo: más
rentabilidad asociada a más riesgo no tiene sentido económico. Es el tipo de resultado que
aparece cuando los ratios están correlacionados entre sí (abajo, la matriz de correlaciones:
`roa` se correlaciona 0,40 con `ventas_deuda` y −0,36 con `deuda_activos`) y la relación no es
lineal: la regresión reparte el efecto conjunto como puede. En la sección 6 vamos a preguntarle
a un modelo sin ese supuesto si opina lo mismo.

**Lo que el coeficiente promete y lo que supone.** Promete una lectura limpia: dirección, tamaño
e incertidumbre. A cambio supone que el efecto de cada ratio es el mismo en todo su rango
(lineal en el log-odds) y que no depende de los otros ratios (aditivo). Cuando la relación tiene
forma de U o de escalón (la del Notebook 1), o cuando "la garantía importa solo si la deuda es
alta" (el Notebook 2), el coeficiente promedia todo eso en un solo número.


In [ ]:

# Cuánto se correlacionan los ratios entre sí (parte de la explicación del signo de roa)
print(X_train.corr().round(2))



## 3. En un árbol: cuánta impureza reduce cada variable

Un árbol no tiene coeficientes, pero en cada nodo la pregunta elegida **reduce la impureza** del
grupo en una cantidad conocida: la ganancia de información en ID3, la caída del Gini en CART, la
caída de la varianza en regresión. La importancia de una variable es la suma de esas reducciones
en todos los nodos donde se preguntó por ella, pesando cada nodo por la fracción de observaciones
que pasó por él, y normalizada para que las importancias sumen 1:

$$
\text{imp}(x_j) \;\propto\; \sum_{t \,:\, x_j \text{ corta en } t} \frac{n_t}{n}
\left[ I(t) - \frac{n_{t,\text{izq}}}{n_t} I(t_{\text{izq}}) - \frac{n_{t,\text{der}}}{n_t} I(t_{\text{der}}) \right]
$$

donde $n_t$ es cuántas observaciones llegan al nodo $t$, $n$ el total, e $I$ la impureza que use
el árbol. El corchete es la reducción de impureza que consigue ese nodo; el factor $n_t / n$
hace que un corte en la raíz, por el que pasan todas, pese más que uno abajo.

Lo calculamos a mano recorriendo la estructura interna del árbol de `scikit-learn`
(`arbol.tree_`) y lo comparamos con `feature_importances_`, que es exactamente esta cuenta.


In [ ]:

arbol2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
t = arbol2.tree_
n = t.n_node_samples[0]

def importancia_a_mano(t, nombres):
    acum = pd.Series(0.0, index=nombres)
    for nodo in range(t.node_count):
        izq, der = t.children_left[nodo], t.children_right[nodo]
        if izq == -1:            # hoja
            continue
        reduccion = t.impurity[nodo] - (t.n_node_samples[izq] / t.n_node_samples[nodo]) * t.impurity[izq] \
                                     - (t.n_node_samples[der] / t.n_node_samples[nodo]) * t.impurity[der]
        acum[nombres[t.feature[nodo]]] += t.n_node_samples[nodo] / t.n_node_samples[0] * reduccion
        print(f"nodo {nodo}: {nombres[t.feature[nodo]]:16s} n = {t.n_node_samples[nodo]:3d}   impureza {t.impurity[nodo]:.3f}   reducción {reduccion:.4f}   ponderada {t.n_node_samples[nodo] / t.n_node_samples[0] * reduccion:.4f}")
    return acum / acum.sum()

imp_mano = importancia_a_mano(t, RATIOS)
print()
comparacion = pd.DataFrame({"a mano": imp_mano, "feature_importances_": arbol2.feature_importances_}).round(3)
print(comparacion)



Las dos columnas coinciden: `feature_importances_` es esta suma. `ventas_deuda` se lleva la
mayor parte porque es la raíz y su corte lo ven las 146 empresas; `ln_activos` y `roa` quedan en
cero porque nunca se preguntó por ellas, aunque tal vez hayan quedado segundas por poco en algún
nodo. Compara con la regresión: allí `ventas_deuda` no era la primera. No es que uno de los dos se
equivoque; miden cosas distintas.

**Lo mismo para ID3, con entropía.** Sobre las 14 empresas del Notebook 2 el árbol tiene tres
nodos internos: deuda en la raíz (14 empresas), garantía en la rama de deuda alta (5) e historial
en la de deuda media (5).


In [ ]:

# Las 14 empresas que pidieron crédito (la tabla chica del curso)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


In [ ]:

def H(serie):
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia(df, atributo):
    return H(df["paga"]) - sum(len(g) / len(df) * H(g["paga"]) for _, g in df.groupby(atributo))

nodos_id3 = [("raíz", credito, "deuda"), ("deuda = alta", credito[credito.deuda == "alta"], "garantia"), ("deuda = media", credito[credito.deuda == "media"], "historial")]
imp_id3 = pd.Series(0.0, index=["tamano", "deuda", "garantia", "historial"])
for nombre, grupo, atributo in nodos_id3:
    g = ganancia(grupo, atributo)
    imp_id3[atributo] += len(grupo) / len(credito) * g
    print(f"{nombre:14s} pregunta {atributo:10s} n = {len(grupo):2d}   ganancia {g:.3f}   ponderada {len(grupo) / len(credito) * g:.3f}")
print()
print((imp_id3 / imp_id3.sum()).round(3))



Fíjate en que garantía e historial superan a deuda, que es la raíz: sus preguntas dejan grupos
puros (ganancia 0,971 cada una) y, pesadas por 5/14, dan 0,347 contra los 0,292 de la raíz. En un
árbol tan chico la raíz no siempre gana. Tamaño nunca se preguntó: importancia cero, aunque en la
raíz tuvo ganancia 0,006.

**Lo que esta importancia dice y lo que no.** Dice cuánto usó el árbol cada variable para ordenar
los datos, con relaciones de cualquier forma e interacciones incluidas. No dice la dirección del
efecto (la misma variable puede subir el riesgo en una rama y bajarlo en otra), no tiene unidades
ni error estándar, y hereda la inestabilidad del árbol. Y tiene un sesgo conocido que vemos en la
sección siguiente.



## 4. En un bosque: promediar sobre los árboles, y un sesgo que conviene conocer

Con muchos árboles la cuenta es la misma en cada uno y se promedia. El promedio arregla la
inestabilidad: una variable que en un árbol quedó en cero aparece en otros.


In [ ]:

bosque = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_train, y_train)
imp_bosque = pd.Series(bosque.feature_importances_, index=RATIOS)
por_arbol = pd.DataFrame([a.feature_importances_ for a in bosque.estimators_], columns=RATIOS)
resumen = pd.DataFrame({"árbol prof. 2": arbol2.feature_importances_, "random forest (promedio de 500)": imp_bosque,
                        "desv. est. entre árboles": por_arbol.std()}).round(3)
print(resumen)
print(f"\nAcierto del bosque en prueba: {accuracy_score(y_test, bosque.predict(X_test)):.3f}")



**El sesgo hacia variables con muchos valores.** Una variable numérica con muchos valores
distintos ofrece muchos más cortes candidatos que una de dos valores, y por eso tiende a ganar
importancia "gratis": es el mismo problema de la trampa del identificador del Notebook 1. Para
verlo, agregamos dos columnas de puro ruido, sin ninguna relación con el impago: una con 146
valores distintos y otra con solo dos.


In [ ]:

rng = np.random.default_rng(5)
X_ruido = X_train.copy(); Xt_ruido = X_test.copy()
X_ruido["ruido_continuo"] = rng.normal(size=len(X_train)); Xt_ruido["ruido_continuo"] = rng.normal(size=len(X_test))
X_ruido["ruido_binario"] = rng.integers(0, 2, len(X_train)); Xt_ruido["ruido_binario"] = rng.integers(0, 2, len(X_test))

bosque_ruido = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_ruido, y_train)
imp_ruido = pd.Series(bosque_ruido.feature_importances_, index=X_ruido.columns).round(3)
print(imp_ruido.sort_values(ascending=False))



El ruido continuo, que no tiene ninguna información, recibe una importancia por impureza
comparable a la de ratios reales; el ruido binario, casi nada. No es que el bosque "crea" que el
ruido continuo sirve: es que, con 146 valores distintos, siempre hay algún corte que ordena un
poco el entrenamiento por casualidad, y la impureza lo cuenta.

### La importancia por permutación

Hay una segunda forma de medir importancia que no mira dentro del modelo y por eso sirve para
cualquiera. Se toma el modelo ya entrenado y las observaciones de prueba, se mide el acierto, y
después se **baraja** una columna (se mezclan sus valores entre las observaciones, rompiendo su
relación con la respuesta) y se vuelve a medir:

$$
\text{imp}_{\text{perm}}(x_j) = \text{acierto}(\text{datos originales}) - \text{acierto}(\text{datos con } x_j \text{ barajada})
$$

Si el acierto cae mucho, el modelo dependía de esa variable; si no cambia, no la estaba usando de
verdad. Como el barajado es aleatorio, se repite varias veces y se promedia. `scikit-learn` lo
trae en `permutation_importance`.


In [ ]:

perm = permutation_importance(bosque_ruido, Xt_ruido, y_test, n_repeats=30, random_state=0, scoring="accuracy")
tabla_perm = pd.DataFrame({"impureza": imp_ruido, "permutación (caída de acierto)": perm.importances_mean, "desv. est. (30 barajados)": perm.importances_std}, index=Xt_ruido.columns).round(3)
print(tabla_perm.sort_values("permutación (caída de acierto)", ascending=False))



Por permutación, las dos columnas de ruido quedan en cero (o levemente negativas, que es lo mismo:
barajarlas no empeora nada). La permutación mide lo que importa en el negocio, cuánto empeora la
predicción si esa información no estuviera, y se calcula sobre datos de prueba, así que no premia
variables que solo sirvieron para memorizar el entrenamiento.

Su punto débil es el mismo de siempre con variables correlacionadas: si barajas una y el modelo
tiene otra casi igual, el acierto casi no cae y las dos parecen poco importantes. Con 73 empresas
de prueba, además, un punto porcentual es menos de una empresa: lo robusto es qué variables están
claramente por encima de cero, no el orden exacto entre ellas.



## 5. En boosting: lo mismo, sumando ronda a ronda

En AdaBoost y en gradient boosting los árboles no se promedian, se suman, pero la importancia se
calcula con la misma idea: se acumula, ronda a ronda, cuánto aportó cada variable. Para AdaBoost
con tocones, `scikit-learn` pesa la importancia de cada tocón por su $\alpha$; para gradient
boosting suma la reducción de impureza de cada árbol sobre todas las rondas.


In [ ]:

ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=100, learning_rate=0.5, random_state=0).fit(X_train, y_train)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=0).fit(X_train, y_train)

# AdaBoost a mano: alfa acumulado por variable (cada tocón usa una sola variable)
alfa_por_variable = pd.Series(0.0, index=RATIOS)
for tocon, alfa in zip(ada.estimators_, ada.estimator_weights_):
    alfa_por_variable[RATIOS[tocon.tree_.feature[0]]] += alfa

tabla_boost = pd.DataFrame({"AdaBoost (alfa acumulado, normalizado)": alfa_por_variable / alfa_por_variable.sum(),
                            "AdaBoost feature_importances_": ada.feature_importances_,
                            "Gradient boosting feature_importances_": gb.feature_importances_}, index=RATIOS).round(3)
print(tabla_boost)
print(f"\nAcierto en prueba: AdaBoost {accuracy_score(y_test, ada.predict(X_test)):.3f}   gradient boosting {accuracy_score(y_test, gb.predict(X_test)):.3f}")



Las dos primeras columnas coinciden: `scikit-learn` pesa por $\alpha$ la importancia por
impureza de cada tocón, y como un tocón usa una sola variable, su importancia es 1 para esa
variable y 0 para las demás; la cuenta se reduce a sumar los $\alpha$. XGBoost reporta tres versiones de esto: *gain* (la reducción de pérdida acumulada, que
corresponde a la ecuación de la sección 3), *cover* (cuántas observaciones pasaron por los nodos
de esa variable) y *weight* (en cuántos nodos apareció). Conviene mirar *gain*; *weight* premia a
las variables con muchos cortes posibles.

Una advertencia particular de boosting: como cada árbol corrige a los anteriores, una variable
puede aparecer mucho en rondas tardías solo para ajustar ruido residual. La importancia por
impureza la cuenta igual; la de permutación sobre datos de prueba, no. Cuando las dos difieren
mucho, suele ser señal de sobreajuste.



## 6. SHAP: recuperar el signo, empresa por empresa

Ninguna de las importancias de árboles tiene signo: dicen que la liquidez importa, no si mucha
liquidez sube o baja el riesgo. Para recuperar la dirección hay dos caminos. Uno es mirar las
hojas, como en los Notebooks 2 y 3: la regla "ventas sobre deuda menor a 1,28 implica más impago"
es una dirección. El otro, para ensambles, son los métodos de **contribución por observación**,
de los que el más usado es SHAP (Lundberg y Lee, 2017).

La idea: la predicción de cada empresa se reparte entre sus variables, de modo que

$$
f(x_i) = \phi_0 + \phi_{i1} + \phi_{i2} + \dots + \phi_{i5}
$$

donde $f(x_i)$ es la predicción del modelo para la empresa $i$ (en log-odds, para un
clasificador), $\phi_0$ es la predicción promedio, y $\phi_{ij}$ es la contribución del ratio
$j$ **para esa empresa**, con signo: positivo si ese ratio empujó la predicción hacia el impago,
negativo si la alejó. El reparto se hace con una regla de la teoría de juegos (el valor de
Shapley) que es la única que cumple algunas propiedades razonables de justicia entre variables.
Al promediar los valores absolutos de $\phi_{ij}$ sobre todas las empresas se obtiene una
importancia global, y al mirar el signo de cada $\phi_{ij}$ contra el valor del ratio se recupera
la dirección.


In [ ]:

try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap

explicador = shap.TreeExplainer(gb)
phi = explicador.shap_values(X_test)                 # una fila por empresa de prueba, una columna por ratio (en log-odds)
phi = pd.DataFrame(phi, columns=RATIOS, index=X_test.index)

imp_shap = phi.abs().mean().sort_values(ascending=False)
direccion = pd.Series({r: np.corrcoef(X_test[r], phi[r])[0, 1] for r in RATIOS})
print(pd.DataFrame({"importancia SHAP (|phi| promedio)": imp_shap.round(3), "correlación entre el ratio y su contribución": direccion[imp_shap.index].round(2),
                    "lectura": np.where(direccion[imp_shap.index] > 0, "más alto, más riesgo", "más alto, menos riesgo")}))


In [ ]:

# El gráfico clásico de SHAP: cada punto es una empresa; el color es el valor del ratio; la posición, su contribución
shap.summary_plot(phi.values, X_test, feature_names=RATIOS, show=True)


In [ ]:

# Una empresa concreta: por qué el gradient boosting le da la probabilidad que le da
i = X_test.index[0]
p_i = gb.predict_proba(X_test.loc[[i]])[0, 1]
print(f"Empresa {i}: impago real = {y_test.loc[i]}, probabilidad de impago según gradient boosting = {p_i:.3f}")
print(f"Predicción promedio (log-odds): {explicador.expected_value[0] if np.ndim(explicador.expected_value) else explicador.expected_value:.3f}")
print()
print(pd.DataFrame({"valor del ratio": X_test.loc[i].round(3), "promedio del entrenamiento": X_train.mean().round(3), "contribución (log-odds)": phi.loc[i].round(3)}).sort_values("contribución (log-odds)", key=abs, ascending=False))



**Sobre el signo de `roa`.** SHAP sobre el gradient boosting también encuentra que más
rentabilidad empuja hacia el impago (correlación positiva entre el ratio y su contribución). Así
que la sorpresa no era un artefacto de la linealidad de la regresión: está en los datos. Con 146
empresas puede ser una casualidad de la muestra, o puede ser real (empresas con rentabilidad alta
pero endeudamiento alto, por ejemplo). Lo que corresponde no es reportarlo ni esconderlo, sino
investigarlo: mirar esas empresas antes de poner el modelo en producción.

**Cómo leerlo.** Para esta empresa, cada ratio empujó la predicción en la dirección y con la
fuerza que dice la columna de contribución; la suma de las contribuciones más la predicción
promedio da el log-odds de la probabilidad que el modelo entregó. Esto es lo que se le muestra a
un regulador cuando pide justificar un rechazo con un ensamble: no una regla, pero sí "estas dos
variables, con estos valores, explican la mayor parte de la decisión". Es lo más parecido a un
coeficiente que tiene un ensamble, con una diferencia: el efecto de un ratio puede ser distinto
para cada empresa, que es justamente lo que la regresión suponía que no pasaba.



## 7. Todo junto

La tabla reúne, para los cinco ratios, las medidas de este notebook, todas con la misma
partición.


In [ ]:

perm_bosque = permutation_importance(bosque, X_test, y_test, n_repeats=30, random_state=0, scoring="accuracy")
todo = pd.DataFrame({
    "beta logístico estandarizado": logit.params.drop("const"),
    "árbol prof. 2 (impureza)": arbol2.feature_importances_,
    "random forest (impureza)": bosque.feature_importances_,
    "gradient boosting (impureza)": gb.feature_importances_,
    "random forest (permutación)": perm_bosque.importances_mean,
    "SHAP sobre gradient boosting": imp_shap[RATIOS],
}, index=RATIOS).round(3)
print(todo)
print()
ranking = todo.copy(); ranking["beta logístico estandarizado"] = ranking["beta logístico estandarizado"].abs()
print("Ranking (1 = la más importante; el beta por valor absoluto):")
print(ranking.rank(ascending=False).astype(int))


In [ ]:

normalizado = todo.drop(columns="beta logístico estandarizado").div(todo.drop(columns="beta logístico estandarizado").sum())
fig = px.bar(normalizado.reset_index().melt(id_vars="index", var_name="medida", value_name="importancia (normalizada a 1)"),
             x="index", y="importancia (normalizada a 1)", color="medida", barmode="group", title="Cinco medidas de importancia, normalizadas para que cada una sume 1")
fig.update_layout(xaxis_title="", height=420)
fig.show()



**Cómo leer las diferencias.** Los números responden preguntas distintas. El coeficiente
responde "si muevo este ratio una desviación estándar y dejo todo lo demás igual, cuánto cambia
el log-odds de impago, en promedio y suponiendo una relación lineal". La importancia por impureza
responde "cuánto usó el modelo este ratio para separar a las empresas de entrenamiento". La de
permutación responde "cuánto acierto pierdo en empresas nuevas si esta información no
estuviera". SHAP responde "cuánto movió este ratio la predicción de cada empresa, en promedio".
Que un ratio sea el primero en una lista y el tercero en otra no es una contradicción, y con 146
empresas todas las listas son ruidosas.

**Para qué sirve.** Para elegir la explicación correcta según a quién se le habla. A un comité de
crédito que exige reglas: un árbol chico y sus hojas. A un gerente que pregunta qué mirar:
importancias del bosque, con la advertencia de que no traen signo. A un regulador que exige
justificar cada rechazo: contribuciones por cliente. Y a un econometrista que ya tiene su
regresión: los betas siguen siendo la lectura más limpia cuando la relación es aproximadamente
lineal, y el ensamble le dice cuándo no lo es.



## Ejercicios

1. **Odds ratio.** Convierte el beta estandarizado de `razon_corriente` en un *odds ratio*
   ($e^{\beta}$) y explícalo en una frase para un gerente: "una desviación estándar más de razón
   corriente multiplica las chances de impago por ...".

2. **La regresión también se puede permutar.** Calcula la importancia por permutación de la
   regresión logística (sobre las 73 de prueba, 30 barajados) y compárala con los betas
   estandarizados. ¿Coinciden los rankings? ¿Debería sorprender que sí o que no?

3. **Correlacionadas.** Agrega a la tabla una columna `deuda_activos_copia` igual a
   `deuda_activos` más un ruido muy chico, entrena un random forest y mira qué pasa con la
   importancia por impureza y por permutación de las dos. Explica el resultado con una frase.

4. **Un árbol de regresión.** Entrena un `DecisionTreeRegressor` de profundidad 2 para predecir
   `roa` a partir de los otros cuatro ratios, y calcula su importancia a mano con la función
   `importancia_a_mano` (la impureza ahora es la varianza). Compárala con los coeficientes
   estandarizados de una regresión lineal de `roa` sobre los mismos cuatro ratios.

5. **SHAP con signo cambiado.** Busca en `phi` una empresa para la que `razon_corriente` tenga
   contribución positiva (empuje hacia el impago) y otra para la que tenga contribución negativa.
   ¿Qué valores del ratio tienen? ¿Es coherente con el signo del beta de la regresión?



## Soluciones


In [ ]:

# 1. Odds ratio
b = logit.params["razon_corriente"]
print(f"beta = {b:.3f}   odds ratio = exp(beta) = {np.exp(b):.3f}")
print(f"Una desviación estándar más de razón corriente ({sd['razon_corriente']:.2f} unidades) multiplica las chances de impago por {np.exp(b):.2f}, es decir, las reduce en un {100 * (1 - np.exp(b)):.0f}%, manteniendo los otros ratios fijos.")


In [ ]:

# 2. Permutar la regresión logística
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
logit_sk = make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=1000)).fit(X_train, y_train)
perm_logit = permutation_importance(logit_sk, X_test, y_test, n_repeats=30, random_state=0, scoring="accuracy")
sol2 = pd.DataFrame({"|beta estandarizado|": logit.params.drop("const").abs(), "permutación (caída de acierto)": perm_logit.importances_mean}, index=RATIOS).round(3)
sol2["ranking beta"] = sol2["|beta estandarizado|"].rank(ascending=False).astype(int); sol2["ranking permutación"] = sol2["permutación (caída de acierto)"].rank(ascending=False).astype(int)
print(sol2)
print("\nEl orden es parecido pero no idéntico: el beta mide el efecto marginal en el log-odds, la permutación mide cuánto cae el ACIERTO en 73 empresas,")
print("que solo cambia cuando una probabilidad cruza el umbral 0,5. Un beta grande sobre una variable con poca dispersión en prueba puede mover pocas decisiones.")


In [ ]:

# 3. Dos variables casi iguales
rng = np.random.default_rng(3)
X_dup = X_train.copy(); Xt_dup = X_test.copy()
X_dup["deuda_activos_copia"] = X_dup["deuda_activos"] + rng.normal(scale=0.001, size=len(X_dup))
Xt_dup["deuda_activos_copia"] = Xt_dup["deuda_activos"] + rng.normal(scale=0.001, size=len(Xt_dup))
bosque_dup = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_dup, y_train)
perm_dup = permutation_importance(bosque_dup, Xt_dup, y_test, n_repeats=30, random_state=0, scoring="accuracy")
sol3 = pd.DataFrame({"impureza": bosque_dup.feature_importances_, "permutación": perm_dup.importances_mean}, index=X_dup.columns).round(3)
print(sol3)
print(f"\nSin la copia, deuda_activos tenía impureza {bosque.feature_importances_[0]:.3f} y permutación {perm_bosque.importances_mean[0]:.3f}.")
print("Con la copia, la impureza se reparte entre las dos (el bosque elige una u otra al azar en cada corte) y la permutación de cada una cae casi a cero:")
print("barajar una no duele porque el modelo tiene la otra. Ninguna de las dos medidas 'sabe' que son la misma información.")


In [ ]:

# 4. Árbol de regresión para roa, contra una regresión lineal
from sklearn.tree import DecisionTreeRegressor
SIN_ROA = [r for r in RATIOS if r != "roa"]
arbol_roa = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train[SIN_ROA], X_train["roa"])
imp_roa = importancia_a_mano(arbol_roa.tree_, SIN_ROA)
Z4 = (X_train[SIN_ROA] - X_train[SIN_ROA].mean()) / X_train[SIN_ROA].std(ddof=0)
ols = sm.OLS(X_train["roa"], sm.add_constant(Z4)).fit()
print()
print(pd.DataFrame({"importancia del árbol (varianza)": imp_roa.round(3), "beta estandarizado (OLS)": ols.params.drop("const").round(4), "valor p": ols.pvalues.drop("const").round(3)}))
print("\nEl árbol y la regresión coinciden en qué ratio manda (ventas_deuda), y la regresión agrega el signo: más ventas sobre deuda, más rentabilidad.")


In [ ]:

# 5. La misma variable, contribuciones de signo opuesto
pos = phi["razon_corriente"].idxmax(); neg = phi["razon_corriente"].idxmin()
print(pd.DataFrame({"razón corriente": X_test.loc[[pos, neg], "razon_corriente"].round(3).values, "contribución SHAP": phi.loc[[pos, neg], "razon_corriente"].round(3).values, "impago real": y_test.loc[[pos, neg]].values}, index=[f"empresa {pos} (empuja al impago)", f"empresa {neg} (aleja del impago)"]))
print(f"\nPromedio de razón corriente en entrenamiento: {X_train['razon_corriente'].mean():.3f}")
print("La empresa con contribución positiva tiene razón corriente baja; la de contribución negativa, alta. Es coherente con el beta negativo de la")
print("regresión (más liquidez, menos riesgo), pero SHAP lo dice empresa por empresa y sin suponer que el efecto sea el mismo en todo el rango.")
